In [ ]:
# Abhängigkeits-Check für den Workshop-Teil – installiert nur, was fehlt
import importlib, importlib.util, subprocess, sys

REQUIRED = {                      # import-Name -> pip-Name
    "numpy": "numpy",
    "networkx": "networkx",
    "matplotlib": "matplotlib",
    "sentence_transformers": "sentence-transformers",
}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installiere fehlende Pakete:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
for mod in REQUIRED:
    importlib.import_module(mod)
print("✅ Zusatz-Libraries importierbar:", ", ".join(REQUIRED))

# Hinweis: tree-sitter-languages enthält keine COBOL-Grammatik (und hat keine Wheels für Python 3.13).
# Das "AST-aware Chunking" in Abschnitt 1 nutzt deshalb einen strukturierten COBOL-Parser
# (Divisions, Paragraphen in Area A, 01/77-Level-Gruppen) – dieselbe Idee, ohne externe Grammatik.

In [ ]:
import subprocess

def ask_claude_code(prompt: str) -> str:
    # -p bzw. --print führt den Prompt aus und gibt das Ergebnis direkt aus
    result = subprocess.run(
        ["claude", "-p", prompt],
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout

antwort = ask_claude_code("Schreibe eine Python-Funktion für Fibonacci-Zahlen.")
print(antwort)

In [1]:
"""Claude Code CLI (`claude -p`) mit Anthropic-Messages-API-kompatiblem Output."""

from __future__ import annotations

import json
import subprocess
import uuid
from typing import Any, Iterator


class ClaudeCodeError(RuntimeError):
    """claude ist mit != 0 beendet oder hat unlesbaren Output geliefert."""


# --- non-streaming ----------------------------------------------------------


def ask_claude_code(
    prompt: str,
    *,
    model: str | None = None,
    allowed_tools: str | None = None,
    json_schema: dict[str, Any] | None = None,
    bare: bool = False,
    timeout: float | None = 300,
) -> dict[str, Any]:
    """Führt einen Prompt aus und gibt ein Messages-API-förmiges dict zurück.

    bare=True überspringt CLAUDE.md, Hooks, MCP-Server und Plugins (schneller,
    reproduzierbar), liest dann aber KEINE OAuth-Credentials und keinen
    System-Keychain -> ANTHROPIC_API_KEY muss gesetzt sein.
    """
    cmd = ["claude", "-p", prompt, "--output-format", "json"]
    if bare:
        cmd.append("--bare")
    if model:
        cmd += ["--model", model]
    if allowed_tools:
        cmd += ["--allowedTools", allowed_tools]
    if json_schema is not None:
        cmd += ["--json-schema", json.dumps(json_schema)]

    proc = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=timeout,
        stdin=subprocess.DEVNULL,  # sonst wartet claude 3s auf stdin und warnt
    )
    if proc.returncode != 0:
        raise ClaudeCodeError(
            f"claude exit={proc.returncode}\n"
            f"stdout: {proc.stdout.strip()!r}\n"
            f"stderr: {proc.stderr.strip()!r}"
        )

    try:
        raw = json.loads(proc.stdout)
    except json.JSONDecodeError as exc:
        raise ClaudeCodeError(f"Kein JSON auf stdout: {proc.stdout[:500]!r}") from exc

    return to_message(raw, fallback_model=model)


_STOP_REASONS = {
    "success": "end_turn",
    "error_max_turns": "max_tokens",  # kein exaktes API-Äquivalent
}


def to_message(raw: dict[str, Any], *, fallback_model: str | None = None) -> dict[str, Any]:
    """Mappt eine Claude-Code-`result`-Message auf das Messages-API-Schema."""
    usage = raw.get("usage") or {}
    model_usage = raw.get("modelUsage") or {}
    model = (
        raw.get("model")
        or next(iter(model_usage), None)
        or fallback_model
        or "claude-code"
    )

    content: list[dict[str, Any]] = []
    text = raw.get("result") or ""
    if text:
        content.append({"type": "text", "text": text})

    message = {
        "id": f"msg_{uuid.uuid4().hex[:24]}",
        "type": "message",
        "role": "assistant",
        "model": model,
        "content": content,
        "stop_reason": _STOP_REASONS.get(raw.get("subtype")),
        "stop_sequence": None,
        "usage": {
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
            "cache_creation_input_tokens": usage.get("cache_creation_input_tokens", 0),
            "cache_read_input_tokens": usage.get("cache_read_input_tokens", 0),
        },
        # nicht im API-Schema, aber meist zu wertvoll zum Wegwerfen:
        "_claude_code": {
            "session_id": raw.get("session_id"),
            "num_turns": raw.get("num_turns"),
            "duration_ms": raw.get("duration_ms"),
            "total_cost_usd": raw.get("total_cost_usd"),
            "is_error": raw.get("is_error"),
            "subtype": raw.get("subtype"),
            "structured_output": raw.get("structured_output"),
            "permission_denials": raw.get("permission_denials"),
        },
    }
    return message


def text_of(message: dict[str, Any]) -> str:
    """Wie data.content bei der echten API: alle text-Blöcke zusammenfügen."""
    return "\n".join(b["text"] for b in message["content"] if b.get("type") == "text")


# --- streaming --------------------------------------------------------------


def stream_claude_code(
    prompt: str,
    *,
    model: str | None = None,
    bare: bool = False,
) -> Iterator[dict[str, Any]]:
    """Yieldet rohe Messages-API-Stream-Events (message_start, content_block_delta, ...).

    Achtung: Claude Code ist eine Agent-Loop, also können mehrere
    message_start/message_stop-Zyklen pro Aufruf kommen.
    """
    cmd = [
        "claude", "-p", prompt,
        "--output-format", "stream-json",
        "--verbose",
        "--include-partial-messages",
    ]
    if bare:
        cmd.append("--bare")
    if model:
        cmd += ["--model", model]

    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        stdin=subprocess.DEVNULL,
        text=True,
    ) as proc:
        for line in proc.stdout:
            line = line.strip()
            if not line:
                continue
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                continue
            if event.get("type") == "stream_event":
                yield event["event"]          # bereits API-kompatibel
            elif event.get("type") == "result":
                yield {"type": "_result", "message": to_message(event, fallback_model=model)}
        if proc.wait() != 0:
            raise ClaudeCodeError(proc.stderr.read().strip())


if __name__ == "__main__":
    msg = ask_claude_code("Schreibe eine Python-Funktion für Fibonacci-Zahlen.")
    print(json.dumps(msg, indent=2, ensure_ascii=False))
    print("---")
    print(text_of(msg))

{
  "id": "msg_fb559a3bef24459caa128647",
  "type": "message",
  "role": "assistant",
  "model": "claude-opus-5[1m]",
  "content": [
    {
      "type": "text",
      "text": "```python\ndef fibonacci(n: int) -> int:\n    \"\"\"Gibt die n-te Fibonacci-Zahl zurück (fib(0) = 0, fib(1) = 1).\"\"\"\n    if n < 0:\n        raise ValueError(\"n muss >= 0 sein\")\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n```\n\nIterativ, damit auch große `n` ohne Rekursionslimit funktionieren — `fibonacci(1000)` läuft sofort durch (Python-Ints haben beliebige Größe).\n\nFalls du stattdessen die Folge brauchst, ist ein Generator praktischer:\n\n```python\nfrom typing import Iterator\n\n\ndef fibonacci_sequence() -> Iterator[int]:\n    \"\"\"Erzeugt die Fibonacci-Folge unendlich: 0, 1, 1, 2, 3, 5, ...\"\"\"\n    a, b = 0, 1\n    while True:\n        yield a\n        a, b = b, a + b\n\n\n# Beispiel: die ersten 10 Zahlen\nfrom itertools import islice\nprint(list(islice(fibon

# Workshop: Make Legacy Codebase Searchable

Drei Live-Demos auf echtem COBOL/CICS-Code aus **AWS CardDemo** (Kreditkarten-Verwaltung: Sign-on, Kartenliste, Transaktionen).

| # | Demo | Leitfrage | Werkzeug |
|---|------|-----------|----------|
| 1 | **Ingestion** | Wie zerlege ich Legacy-Code, ohne die Semantik zu zerstören? | Struktur-Parser (Paragraphen, 01-Level-Gruppen) |
| 2 | **Retrieval** | Welches Programm *schreibt* Transaktionsdaten? | `grep` vs. Embeddings + Cosine Similarity |
| 3 | **Code-Intelligence** | Was passiert, wenn ich Copybook X ändere? | Knowledge Graph mit `networkx` |

**Bausteine aus diesem Notebook**

- LLM-Aufrufe: `ask_claude_code(prompt, ...) -> dict` (Messages-API-Format) und `text_of(message) -> str` – oben definiert.
- Embeddings: `embed_texts(texts) -> np.ndarray` – wird im Setup ergänzt (lokales Sentence-Transformer-Modell, keine API nötig).

**Hinweise zur Datenbasis**

- CardDemo enthält kein `COTRN01Y.cpy`. Der Transaktionssatz (`TRAN-RECORD`) liegt in **`CVTRA05Y.cpy`** – dieses Copybook wird von allen drei Transaktionsprogrammen eingebunden und ist daher die zweite Copybook-Datei.
- Alle Dateien werden nach `data/` neben dieses Notebook kopiert; das Original-Repo bleibt unangetastet.

## Setup: Datenbasis `data/` und Embedding-Helfer

Kopiert 5 Programme + 2 Copybooks aus CardDemo nach `data/` und definiert `embed_texts()`.
Das Embedding-Modell wird beim ersten Aufruf geladen (≈ 10–15 s, danach gecacht).

In [ ]:
import html as _html
import os, re, shutil
from pathlib import Path

import numpy as np
from IPython.display import display, HTML, Markdown

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# --- 1) CardDemo lokalisieren: Notebook-Ordner und dessen Eltern durchsuchen --------------------
def find_carddemo(start: Path = Path.cwd()) -> Path:
    for base in [start, *start.parents]:
        for cand in sorted(base.glob("*carddemo*")):
            if (cand / "app" / "cbl").is_dir():
                return cand
    raise FileNotFoundError("CardDemo-Ordner (…/app/cbl) nicht gefunden – Notebook im Projekt-Root starten")

CARDDEMO = find_carddemo()
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)

# --- 2) Repräsentative Auswahl kopieren: 5 CICS-Programme + 2 zentrale Copybooks ----------------
SELECTION = {
    "cbl": ["COCRDLIC.cbl",   # Kartenliste (großes Programm, viele EVALUATEs)
            "COTRN00C.cbl",   # Transaktionsliste (nur lesen / browsen)
            "COTRN01C.cbl",   # Transaktion anzeigen (nur lesen)
            "COTRN02C.cbl",   # Transaktion anlegen -> EXEC CICS WRITE
            "COSGN00C.cbl"],  # Sign-on
    "cpy": ["COCOM01Y.cpy",   # CARDDEMO-COMMAREA: von jedem Programm eingebunden
            "CVTRA05Y.cpy"],  # TRAN-RECORD: Satzaufbau der Transaktionsdatei
}
for sub, names in SELECTION.items():
    for name in names:
        shutil.copy2(CARDDEMO / "app" / sub / name, DATA_DIR / name)

DATA_FILES = sorted(DATA_DIR.glob("*.cbl")) + sorted(DATA_DIR.glob("*.cpy"))
print(f"CardDemo : {CARDDEMO}")
print(f"data/    : {DATA_DIR}  ({len(DATA_FILES)} Dateien)\n")
for f in DATA_FILES:
    n_lines = sum(1 for _ in f.open(errors="replace"))
    print(f"  {f.name:<14} {n_lines:>5} Zeilen")

# --- 3) Embedding-Helfer – Gegenstück zu ask_claude_code()/text_of() ----------------------------
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-large-en-v1.5"   # lokal gecacht; all-MiniLM-L6-v2 wäre schneller, rankt COBOL aber schlechter
_embedder: SentenceTransformer | None = None

def embed_texts(texts: list[str]) -> np.ndarray:
    """Liefert L2-normalisierte Embeddings (n × d). Cosine Similarity ist dann ein Skalarprodukt."""
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
    return _embedder.encode(list(texts), normalize_embeddings=True, batch_size=16, show_progress_bar=False)

print(f"\nembed_texts() bereit – Modell: {EMBED_MODEL}")

## 1 · The Ingestion Problem – Naives vs. AST-aware Chunking

Bevor Code durchsuchbar wird, muss er in Stücke ("Chunks") zerlegt werden. Der naive Weg – **alle 45 Zeilen ein Schnitt** – ignoriert, dass COBOL-Code aus **geschlossenen Sinneinheiten** besteht:

- Ein `EVALUATE … WHEN … END-EVALUATE`-Block (das COBOL-`switch`) wird mitten in einem `WHEN`-Zweig durchtrennt. Der Chunk enthält Bedingungen ohne Folgen – oder Folgen ohne Bedingungen.
- Eine hierarchische Datenstruktur (`01 → 05 → 07 → 88`) verliert ihren `01`-Eltern. Die Felder `WS-RESP-CD`, `WS-INPUT-FLAG` … stehen dann kontextlos im Raum.
- Der Chunk weiß nicht mehr, **in welchem Paragraph** er steht – genau das ist aber die Adresse, die ein Mensch (oder ein LLM) beim Antworten braucht.

**AST-aware Chunking** nutzt die Struktur der Sprache: In COBOL sind Paragraphen (Name in Area A, Spalte 8, endet mit Punkt) und `01`-Level-Gruppen natürliche, in sich geschlossene Einheiten. Jeder Chunk bekommt Programm, Division, Namen und Zeilenbereich als Kontext mit.

In [ ]:
from dataclasses import dataclass

# ============================ COBOL-Struktur-Parser ("AST light") ==================================
@dataclass
class Chunk:
    program: str      # z. B. COTRN02C
    kind: str         # PARAGRAPH | DATA-GROUP
    name: str         # Paragraph-Name bzw. Name der 01/77-Level-Gruppe
    start: int        # erste Zeile (1-basiert), inkl. direkt vorangehendem Kommentarblock
    end: int          # letzte Zeile
    code: str
    division: str = ""

    @property
    def ref(self) -> str:
        return f"{self.program}::{self.name}"

def cobol_lines(text: str) -> list[str]:
    """Fixed-Format: Spalten 73-80 (Sequence Area) abschneiden, Zeilen rechts trimmen."""
    return [ln[:72].rstrip() for ln in text.splitlines()]

def is_comment(line: str) -> bool:
    return len(line) > 6 and line[6] in "*/"          # Spalte 7 = Indikator

PARA_RE  = re.compile(r"^([A-Z0-9][A-Z0-9-]*)\s*\.\s*$")   # beginnt in Area A (Spalte 8), endet mit Punkt
DIV_RE   = re.compile(r"^\s*([A-Z-]+)\s+DIVISION\s*\.")
LEVEL_RE = re.compile(r"^\s?(01|77)\s+([A-Z0-9-]+)")

def parse_cobol(text: str, program: str) -> list[Chunk]:
    lines = cobol_lines(text)
    chunks: list[Chunk] = []
    division, cur, lead = "", None, []      # lead = Zeilennummern eines laufenden Kommentarblocks

    def close(end: int):
        nonlocal cur
        if cur is not None:
            cur.end, cur.code = end, "\n".join(lines[cur.start - 1:end])
            chunks.append(cur)
            cur = None

    for i, line in enumerate(lines, start=1):
        if is_comment(line):
            lead.append(i)
            continue
        body = line[7:]                                   # Area A beginnt in Spalte 8
        if m := DIV_RE.match(body):
            close(i - 1); division = m.group(1); lead = []
            continue
        start = lead[0] if lead and lead[-1] == i - 1 else i    # Kommentar-Header gehört zum Element
        if division == "PROCEDURE":
            if (m := PARA_RE.match(body)) and "SECTION" not in body:
                close(i - 1); cur = Chunk(program, "PARAGRAPH", m.group(1), start, 0, "", division)
        elif m := LEVEL_RE.match(body):
            close(i - 1); cur = Chunk(program, "DATA-GROUP", m.group(2), start, 0, "", division or "COPYBOOK")
        lead = []
    close(len(lines))
    return chunks

# ============================ Diagnose: Ist ein Chunk "in sich geschlossen"? =======================
EVAL_OPEN  = re.compile(r"(?<!END-)\bEVALUATE\b")
EVAL_CLOSE = re.compile(r"\bEND-EVALUATE\b")

def diagnose(block: list[str]) -> dict:
    code = [l for l in block if not is_comment(l) and l.strip()]
    joined = "\n".join(code)
    first = code[0][7:] if code else ""
    clean_start = bool(DIV_RE.match(first) or PARA_RE.match(first) or LEVEL_RE.match(first)
                       or "SECTION" in first)
    lvl = re.match(r"^\s*(\d\d)\s", first)
    return {
        "eval_open":  len(EVAL_OPEN.findall(joined)),
        "eval_close": len(EVAL_CLOSE.findall(joined)),
        "clean_start": clean_start,
        "orphan_level": lvl.group(1) if lvl and lvl.group(1) not in ("01", "77") else None,
    }

# ============================ Naives Chunking: feste 45 Zeilen ======================================
SRC = DATA_DIR / "COCRDLIC.cbl"
lines = cobol_lines(SRC.read_text(errors="replace"))
CHUNK = 45
naive = [(i + 1, lines[i:i + CHUNK]) for i in range(0, len(lines), CHUNK)]   # (Startzeile, Zeilen)

# AST-aware Chunking: Paragraphen + 01-Gruppen
ast_chunks = parse_cobol(SRC.read_text(errors="replace"), SRC.stem)
paragraph_of = {}                                  # Zeile -> Paragraph-/Gruppenname
for c in ast_chunks:
    for ln in range(c.start, c.end + 1):
        paragraph_of[ln] = c.name

# Beispiel-Chunks auswählen: (a) Datenblock ohne 01-Eltern, (b) offenes EVALUATE
idx_data = next(i for i, (_, b) in enumerate(naive) if diagnose(b)["orphan_level"])
idx_eval = next(i for i, (_, b) in enumerate(naive)
                if (d := diagnose(b))["eval_open"] > d["eval_close"])
ast_example = next(c for c in ast_chunks if c.name == "1400-SETUP-MESSAGE")
callers = [f"Zeile {no}" for no, l in enumerate(lines, 1)
           if re.search(r"PERFORM\s+1400-SETUP-MESSAGE", l) and not is_comment(l)]

# ============================ Darstellung ==========================================================
CSS = """
<style>
.wk-row  {display:flex; gap:14px; align-items:flex-start; flex-wrap:wrap; font-family:system-ui,sans-serif;}
.wk-card {flex:1 1 380px; border:1px solid #d9d8d3; border-radius:8px; background:#fcfcfb; color:#0b0b0b; overflow:hidden;}
.wk-head {padding:8px 12px; font-weight:600; color:#fff;}
.wk-bad  {background:#e34948;}  .wk-good {background:#008300;}
.wk-meta {padding:6px 12px; font-size:12.5px; color:#52514e; border-bottom:1px solid #eceae4;}
.wk-meta li {margin:2px 0;}
.wk-card pre {margin:0; padding:10px 12px; font-size:11.5px; line-height:1.35; max-height:560px; overflow:auto; background:#fcfcfb; color:#0b0b0b;}
.ln  {color:#9a988f; user-select:none;}
.cut {display:block; color:#e34948; font-weight:700; background:#fdecec; margin:2px -12px; padding:2px 12px;}
.hl  {background:#fff3c4;}  .ok {background:#e2f5e2;}
</style>
"""

def panel(title, meta: list[str], code: list[str], first_no: int, good: bool,
          cut_top=False, cut_bottom=False, mark=lambda ln, l: ""):
    rows = []
    if cut_top:
        rows.append('<span class="cut">✂ Chunk-Grenze – der Kontext DAVOR fehlt</span>')
    for i, l in enumerate(code):
        cls = mark(first_no + i, l)
        rows.append(f'<span class="{cls}"><span class="ln">{first_no + i:>5}</span>  {_html.escape(l)}</span>')
    if cut_bottom:
        rows.append('<span class="cut">✂ Chunk-Grenze – hier hart abgeschnitten</span>')
    meta_html = "".join(f"<li>{m}</li>" for m in meta)
    return (f'<div class="wk-card"><div class="wk-head {"wk-good" if good else "wk-bad"}">{title}</div>'
            f'<ul class="wk-meta">{meta_html}</ul><pre>' + "\n".join(rows) + "</pre></div>")

# (a) naiver Chunk mitten in einer Datenstruktur
s_a, b_a = naive[idx_data]; d_a = diagnose(b_a)
panel_a = panel(
    f"❌ Naiv: Chunk #{idx_data + 1} – Zeilen {s_a}–{s_a + len(b_a) - 1}",
    [f"Beginnt mit Level <b>{d_a['orphan_level']}</b> – die <b>01</b>-Elterngruppe "
     f"(<code>{paragraph_of.get(s_a, '?')}</code>, ab Zeile {next(c.start for c in ast_chunks if c.name == paragraph_of.get(s_a))}) fehlt",
     "Welcher Struktur gehören <code>WS-RESP-CD</code>, <code>WS-INPUT-FLAG</code> an? → nicht rekonstruierbar"],
    b_a[:16] + ["      *  … (weitere 29 Zeilen) …"], s_a, good=False, cut_top=True,
    mark=lambda ln, l: "hl" if re.match(r"^\s{8,}0[5-9]\s|^\s{8,}[1-4]\d\s|^\s{8,}88\s", l) and ln < s_a + 3 else "")

# (b) naiver Chunk mit offenem EVALUATE
s_b, b_b = naive[idx_eval]; d_b = diagnose(b_b)
panel_b = panel(
    f"❌ Naiv: Chunk #{idx_eval + 1} – Zeilen {s_b}–{s_b + len(b_b) - 1}",
    [f"Beginnt mitten im Paragraph <code>{paragraph_of.get(s_b, '?')}</code> (kein Header im Chunk)",
     f"<b>EVALUATE</b> geöffnet: {d_b['eval_open']} · <b>END-EVALUATE</b>: {d_b['eval_close']} → "
     f"Verzweigung nach {len(b_b)} Zeilen abgeschnitten, die restlichen <code>WHEN</code>-Zweige liegen im nächsten Chunk"],
    b_b, s_b, good=False, cut_top=True, cut_bottom=True,
    mark=lambda ln, l: "hl" if re.search(r"\bEVALUATE\b|^\s+WHEN\b", l) and not is_comment(l) else "")

# (c) AST-Chunk: ein geschlossener Paragraph mit vollständigem EVALUATE
d_c = diagnose(ast_example.code.splitlines())
panel_c = panel(
    f"✅ AST-aware: Paragraph {ast_example.ref} – Zeilen {ast_example.start}–{ast_example.end}",
    [f"Kontext: Programm <b>{ast_example.program}</b> · {ast_example.division} DIVISION · "
     f"{ast_example.end - ast_example.start + 1} Zeilen · aufgerufen via PERFORM in {', '.join(callers)}",
     f"<b>EVALUATE</b> {d_c['eval_open']} / <b>END-EVALUATE</b> {d_c['eval_close']} → vollständig · "
     f"Beginn am Paragraph-Header ✔ · Ende vor dem nächsten Paragraph ✔"],
    ast_example.code.splitlines(), ast_example.start, good=True,
    mark=lambda ln, l: "ok" if PARA_RE.match(l[7:]) or re.search(r"\bEVALUATE\b", l) else "")

display(HTML(CSS + f'<div class="wk-row">{panel_a}{panel_b}</div><div style="height:14px"></div>'
                   f'<div class="wk-row">{panel_c}</div>'))

# ============================ Zahlen für die ganze Datei ===========================================
n_bad_eval   = sum(1 for _, b in naive if (d := diagnose(b))["eval_open"] != d["eval_close"])
n_bad_start  = sum(1 for _, b in naive if not diagnose(b)["clean_start"])
a_bad_eval   = sum(1 for c in ast_chunks if (d := diagnose(c.code.splitlines()))["eval_open"] != d["eval_close"])
display(Markdown(f"""
**{SRC.name}: {len(lines)} Zeilen**

| Methode | Chunks | … mit unvollständigem `EVALUATE` | … die mitten in Paragraph/Datenblock beginnen |
|---|---:|---:|---:|
| Naiv (feste {CHUNK} Zeilen) | {len(naive)} | **{n_bad_eval}** | **{n_bad_start}** |
| AST-aware (Paragraphen + 01-Gruppen) | {len(ast_chunks)} | {a_bad_eval} | 0 |
"""))

## 2 · Retrieval Battle – `grep` vs. Semantische Suche

**Fachfrage:** *„Welches Programm schreibt oder aktualisiert Transaktionsdaten?“*

- **Weg A – Volltext:** `grep -i TRANSACT data/*` – so beginnt jede Legacy-Analyse. Das Wort steht aber in Dateinamen (`WS-TRANSACT-FILE`), Kommentaren, Paragraph-Labels, `READ`-Statements und Meldungstexten. Die Trefferliste beantwortet nicht *was passiert*, sondern nur *wo das Wort vorkommt*.
- **Weg B – Semantik:** Wir embedden die AST-Chunks aus Abschnitt 1 (alle Programme + Copybooks) und suchen mit dem natürlichsprachlichen Query **„write or update transaction record“**. Cosine Similarity zwischen Query- und Chunk-Vektor liefert das Ranking – ohne dass der Query ein einziges Token mit dem Code teilen muss.

In [ ]:
from collections import Counter

# ============================ Weg A: grep / Regex nach "TRANSACT" ==================================
PATTERN = re.compile(r"TRANSACT", re.IGNORECASE)
WRITE_VERB = re.compile(r"EXEC\s+CICS\s+(RE)?WRITE\b|^\s*(RE)?WRITE\s", re.IGNORECASE)

def classify(line: str) -> str:
    if is_comment(line):
        return "Kommentar"
    body = line[7:].upper()
    if PARA_RE.match(body):
        return "Paragraph-Label"
    if "PERFORM" in body:
        return "PERFORM-Aufruf"
    if "DATASET" in body:
        return "DATASET(…) in EXEC CICS – READ oder WRITE? unklar"
    if re.search(r"\bPIC\b|^\s*\d\d\s", body):
        return "Datendefinition"
    if re.search(r"\b(MOVE|STRING|DISPLAY)\b", body) or "'" in body:
        return "MOVE / Meldungstext"
    return "Sonstiges"

grep_hits = []                                      # (Datei, Zeile, Kategorie, Text)
for f in DATA_FILES:
    for no, line in enumerate(cobol_lines(f.read_text(errors="replace")), start=1):
        if PATTERN.search(line):
            grep_hits.append((f.name, no, classify(line), line.strip()))

by_file = Counter(h[0] for h in grep_hits)
by_cat  = Counter(h[2] for h in grep_hits)
hits_with_write_verb = [h for h in grep_hits if WRITE_VERB.search(h[3]) and h[2] not in ("Kommentar", "Paragraph-Label", "PERFORM-Aufruf")]

print(f"grep -i TRANSACT data/*   →  {len(grep_hits)} Treffer in {len(by_file)} Dateien\n")
print("Treffer pro Datei:")
for name, n in by_file.most_common():
    print(f"  {name:<14} {n:>3}  {'█' * n}")
print("\nWas steckt in den Treffern?")
for cat, n in by_cat.most_common():
    print(f"  {n:>3}  {cat}")
print(f"\nTreffer, die den Schreibbefehl selbst (EXEC CICS WRITE/REWRITE) enthalten: {len(hits_with_write_verb)}")
print("→ Das Verb steht eine Zeile ÜBER dem Dateinamen; grep sieht 'DATASET (WS-TRANSACT-FILE)' bei READ und WRITE identisch.\n")
print("Stichprobe:")
for name, no, cat, text in grep_hits[::max(1, len(grep_hits) // 10)][:10]:
    print(f"  {name}:{no:<5} {text[:60]:<60}  [{cat}]")

# ============================ Weg B: Semantische Vektorsuche =======================================
all_chunks: list[Chunk] = []
for f in DATA_FILES:
    all_chunks += parse_cobol(f.read_text(errors="replace"), f.stem)

QUERY = "write or update transaction record"
doc_vecs = embed_texts([c.code for c in all_chunks])         # (n_chunks × d), normalisiert
q_vec    = embed_texts([QUERY])[0]
scores   = doc_vecs @ q_vec                                    # Cosine Similarity
ranking  = np.argsort(-scores)

def main_verb(c: Chunk) -> str:
    m = re.search(r"EXEC\s+CICS\s+([A-Z]+)", c.code)
    return f"EXEC CICS {m.group(1)}" if m else ("Datenstruktur" if c.kind == "DATA-GROUP" else "–")

rows = []
for rank, i in enumerate(ranking[:5], start=1):
    c = all_chunks[i]
    rows.append(f"| {rank} | {scores[i]:.3f} | `{c.ref}` | {c.kind} | {c.start}–{c.end} | {main_verb(c)} |")
display(Markdown(f"""
**Semantische Suche** über {len(all_chunks)} AST-Chunks aus {len(DATA_FILES)} Dateien · Query: *„{QUERY}“* · Modell: `{EMBED_MODEL}`

| Rang | Cosine | Chunk | Art | Zeilen | zentrale Operation |
|---:|---:|---|---|---|---|
""" + "\n".join(rows)))

top = all_chunks[ranking[0]]
top_is_write = bool(re.search(r"EXEC\s+CICS\s+(RE)?WRITE", top.code))
display(HTML(CSS + panel(
    f"🎯 Top-1: {top.ref} – Zeilen {top.start}–{top.end}",
    [f"Programm <b>{top.program}</b> · {top.kind} · Cosine {scores[ranking[0]]:.3f}",
     "Enthält den Schreibbefehl <code>EXEC CICS WRITE … FROM (TRAN-RECORD)</code> ✔" if top_is_write
     else "Enthält keinen Schreibbefehl ✘"],
    top.code.splitlines(), top.start, good=top_is_write,
    mark=lambda ln, l: "ok" if re.search(r"EXEC\s+CICS\s+(RE)?WRITE|FROM\s+\(TRAN-RECORD\)", l) else "")))

# ============================ Vergleichs-Panel =====================================================
false_positives = len(grep_hits) - len(hits_with_write_verb)
display(HTML(CSS + f"""
<div class="wk-row">
  <div class="wk-card"><div class="wk-head wk-bad">Weg A · grep -i TRANSACT</div>
    <ul class="wk-meta" style="font-size:14px">
      <li><b style="font-size:26px">{len(grep_hits)}</b> Treffer in {len(by_file)} Dateien</li>
      <li><b>{false_positives}</b> davon ohne Schreibbefehl (False Positives)</li>
      <li>Antwort auf die Fachfrage: <b>keine</b> – {len(grep_hits)} Zeilen manuell sichten</li>
    </ul></div>
  <div class="wk-card"><div class="wk-head wk-good">Weg B · Embeddings + Cosine Similarity</div>
    <ul class="wk-meta" style="font-size:14px">
      <li><b style="font-size:26px">Top 1</b> = <code>{top.ref}</code></li>
      <li>Precision@1: <b>{"100 %" if top_is_write else "0 %"}</b> – der Chunk enthält <code>EXEC CICS WRITE</code></li>
      <li>Query teilt kein Token mit dem Code (kein „TRANSACT“, kein „CICS“)</li>
    </ul></div>
</div>"""))

In [ ]:
# Bonus (optional, live zuschaltbar): Den Top-Treffer vom LLM fachlich erklären lassen.
# Nutzt ask_claude_code()/text_of() vom Notebook-Anfang – ruft die Claude-Code-CLI auf (≈ 10–20 s).
RUN_LLM_BONUS = False

if RUN_LLM_BONUS:
    prompt = (
        "Du bist Mainframe-Analyst. Erkläre in maximal 4 Sätzen auf Deutsch, was dieser COBOL/CICS-Paragraph "
        f"fachlich tut und welche Datei er verändert. Programm {top.program}, Paragraph {top.name}:\n\n{top.code}"
    )
    msg = ask_claude_code(prompt)
    display(Markdown(f"**LLM-Erklärung zu `{top.ref}`**\n\n" + text_of(msg)))
else:
    print("Bonus übersprungen (RUN_LLM_BONUS = False). Im Workshop auf True setzen, um ask_claude_code() live zu zeigen.")

## 3 · Beyond Search – Knowledge Graph & Blast-Radius

Suche beantwortet **„Wo steht was?“**. Code-Intelligence beantwortet **„Was passiert, wenn ich X ändere?“** – die Frage, die vor jedem Release, jeder Migration und jedem Copybook-Refactoring gestellt wird.

Dafür extrahieren wir die *Beziehungen* zwischen den Artefakten und legen sie in einen gerichteten Graphen:

- `COPY XYZ` → Copybook `XYZ` wird zur Compile-Zeit in das Programm eingefügt. Ändert sich das Copybook, muss jedes einbindende Programm neu übersetzt und getestet werden.
- `CALL 'XYZ'` → statischer Unterprogramm-Aufruf. Ändert sich die Schnittstelle des Callee, sind alle Caller betroffen.

Kantenrichtung: **Abhängigkeit → Abhängiger** („eine Änderung fließt entlang der Pfeile“). Der **Blast-Radius** einer Änderung ist dann einfach die Menge der Nachfolger (`networkx.descendants`).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================ Beziehungen extrahieren ==============================================
COPY_RE = re.compile(r"^\s*COPY\s+'?([A-Z0-9-]+)'?", re.IGNORECASE)
CALL_RE = re.compile(r"\bCALL\s+'([A-Z0-9-]+)'", re.IGNORECASE)

G = nx.DiGraph()
for f in DATA_FILES:
    G.add_node(f.stem, kind="Copybook" if f.suffix.lower() == ".cpy" else "Programm", in_data=True)

for f in DATA_FILES:
    for no, line in enumerate(cobol_lines(f.read_text(errors="replace")), start=1):
        if is_comment(line):
            continue
        body = line[7:]
        if m := COPY_RE.match(body):
            dep = m.group(1).upper()
            if dep not in G:
                G.add_node(dep, kind="Copybook", in_data=False)
            G.add_edge(dep, f.stem, rel="COPY", line=no)
        for m in CALL_RE.finditer(body):
            dep = m.group(1).upper()
            if dep not in G:
                G.add_node(dep, kind="Programm", in_data=False)
            G.add_edge(dep, f.stem, rel="CALL", line=no)

n_copy = sum(1 for *_, d in G.edges(data=True) if d["rel"] == "COPY")
n_call = sum(1 for *_, d in G.edges(data=True) if d["rel"] == "CALL")
print(f"Knowledge Graph: {G.number_of_nodes()} Knoten · {n_copy} COPY-Kanten · {n_call} CALL-Kante{"n" if n_call != 1 else ""}")
print(f"  Programme in data/ : {sorted(n for n, d in G.nodes(data=True) if d['kind'] == 'Programm' and d['in_data'])}")
print(f"  Copybooks in data/ : {sorted(n for n, d in G.nodes(data=True) if d['kind'] == 'Copybook' and d['in_data'])}")
print(f"  nur referenziert   : {sorted(n for n, d in G.nodes(data=True) if not d['in_data'])}\n")

# ============================ Frage: Was ist betroffen, wenn CVTRA05Y geändert wird? ===============
CHANGED = "CVTRA05Y"                                  # TRAN-RECORD – der Transaktionssatz
affected = nx.descendants(G, CHANGED)

def ascii_tree(g: nx.DiGraph, node: str, prefix: str = "", seen: set | None = None) -> list[str]:
    seen = seen if seen is not None else set()
    out = []
    children = sorted(g.successors(node))
    for i, child in enumerate(children):
        last = i == len(children) - 1
        e = g.edges[node, child]
        tag = "" if g.nodes[child]["in_data"] else "  (nicht in data/)"
        out.append(f"{prefix}{'└── ' if last else '├── '}{child}  [{e['rel']} in Zeile {e['line']} · {g.nodes[child]['kind']}]{tag}")
        if child not in seen:
            seen.add(child)
            out += ascii_tree(g, child, prefix + ("    " if last else "│   "), seen)
    return out

print(f"Blast-Radius für Änderung an {CHANGED} ({G.nodes[CHANGED]['kind']}):  {len(affected)} betroffene Programme\n")
print(f"{CHANGED}  ({G.nodes[CHANGED]['kind']}, TRAN-RECORD)")
print("\n".join(ascii_tree(G, CHANGED)))

# Zum Vergleich: Blast-Radius aller Copybooks in der Datenbasis
rows = []
for cb in sorted((n for n, d in G.nodes(data=True) if d["kind"] == "Copybook"),
                 key=lambda n: -len(nx.descendants(G, n))):
    desc = sorted(nx.descendants(G, cb))
    rows.append(f"| `{cb}` | {'✔' if G.nodes[cb]['in_data'] else '–'} | {len(desc)} | {', '.join(desc)} |")
display(Markdown("\n**Blast-Radius aller referenzierten Copybooks**\n\n| Copybook | in data/ | betroffene Programme | welche |\n|---|:-:|---:|---|\n" + "\n".join(rows)))

# ============================ Plot: zwei Spalten (Abhängigkeiten -> Programme) =====================
COLORS = {"Programm": "#2a78d6", "Copybook": "#eb6834", "changed": "#e34948", "edge": "#b8b6ae", "text": "#0b0b0b"}
left  = sorted((n for n in G if G.in_degree(n) == 0), key=lambda n: (-len(nx.descendants(G, n)), n))
right = sorted(n for n in G if G.in_degree(n) > 0)
pos = {n: (0, -i) for i, n in enumerate(left)}
pos.update({n: (1, -i * (len(left) - 1) / max(1, len(right) - 1)) for i, n in enumerate(right)})

fig, ax = plt.subplots(figsize=(11, 8.5))
fig.patch.set_facecolor("#fcfcfb"); ax.set_facecolor("#fcfcfb")
hot_edges = [(u, v) for u, v in G.edges if u == CHANGED or (u in affected and v in affected)]
cold_edges = [e for e in G.edges if e not in hot_edges]
nx.draw_networkx_edges(G, pos, edgelist=cold_edges, ax=ax, edge_color=COLORS["edge"], width=1, arrows=True,
                       arrowsize=10, arrowstyle="-|>", connectionstyle="arc3,rad=0.05", min_source_margin=12, min_target_margin=14)
nx.draw_networkx_edges(G, pos, edgelist=hot_edges, ax=ax, edge_color=COLORS["changed"], width=2.2, arrows=True,
                       arrowsize=14, arrowstyle="-|>", connectionstyle="arc3,rad=0.05", min_source_margin=12, min_target_margin=14)
for n, d in G.nodes(data=True):
    fill = COLORS["changed"] if n == CHANGED else COLORS[d["kind"]]
    ax.scatter(*pos[n], s=170, color=fill if d["in_data"] else "#fcfcfb", edgecolors=fill, linewidths=2, zorder=3)
    if n in affected:
        ax.scatter(*pos[n], s=420, facecolors="none", edgecolors=COLORS["changed"], linewidths=2, zorder=2)
    ax.text(pos[n][0] + (-0.04 if pos[n][0] == 0 else 0.04), pos[n][1], n, ha="right" if pos[n][0] == 0 else "left",
            va="center", fontsize=9.5, color=COLORS["text"], fontweight="bold" if n == CHANGED or n in affected else "normal")
ax.set_xlim(-0.45, 1.45); ax.axis("off")
ax.set_title(f"Blast-Radius: Änderung an {CHANGED} betrifft {len(affected)} Programm(e)", loc="left", fontsize=13, color=COLORS["text"])
ax.legend(handles=[
    Line2D([], [], marker="o", ls="", ms=9, color=COLORS["Copybook"], label="Copybook"),
    Line2D([], [], marker="o", ls="", ms=9, color=COLORS["Programm"], label="Programm"),
    Line2D([], [], marker="o", ls="", ms=9, markerfacecolor="#fcfcfb", color=COLORS["Programm"], label="nur referenziert (nicht in data/)"),
    Line2D([], [], marker="o", ls="", ms=9, color=COLORS["changed"], label=f"geändert: {CHANGED}"),
    Line2D([], [], marker="o", ls="", ms=12, markerfacecolor="none", color=COLORS["changed"], label="betroffen (Blast-Radius)"),
    Line2D([], [], color=COLORS["edge"], lw=1, label="COPY / CALL  (Abhängigkeit → Abhängiger)"),
], loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, fontsize=9)
plt.tight_layout(); plt.show()